# API Fetching Notebook

## Import 

In [1]:
import requests
import json
import pandas as pd
from datetime import datetime, timezone

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

## Config

In [2]:
WALLETS = {
    "rn1":      "0x2005d16a84ceefa912d4e380cd32e7ff827875ea",
    "sovereign": "0xee613b3fc183ee44f9da9c05f53e2da107e3debf",
    "swisstony": "0x204f72f35326db932158cba6adff0b9a1da95e14",
}

DATA_API  = "https://data-api.polymarket.com"   # positions, trades, activity
GAMMA_API = "https://gamma-api.polymarket.com"  # profiles, market metadata

## HTTP Caller

In [3]:
import time

def get(base_url, endpoint, params=None, retries=3, backoff=2.0):
    """
    GET request with simple retry on transient connection errors.

    retries: number of attempts before giving up
    backoff: seconds to wait between attempts (doubles each retry)
    """
    url = f"{base_url}{endpoint}"
    wait = backoff

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, timeout=30)
            if not response.ok:
                raise RuntimeError(f"HTTP {response.status_code}: {response.text}")
            return response.json()

        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            if attempt == retries:
                raise  # give up after final attempt
            print(f"  ⚠ connection error (attempt {attempt}/{retries}), retrying in {wait}s...")
            time.sleep(wait)
            wait *= 2  # exponential backoff: 2s → 4s → 8s

## Fetch Exploration

### Position

In [4]:
positions = {}

for name, address in WALLETS.items():
    data = get(DATA_API, "/positions", params={"user": address, "limit": 500})
    positions[name] = data
    print(f"\n=== {name} — {len(data)} open positions ===")
    if data:
        print("Fields:", list(data[0].keys()))
        print("First record:")
        print(json.dumps(data[0], indent=2))


=== rn1 — 500 open positions ===
Fields: ['proxyWallet', 'asset', 'conditionId', 'size', 'avgPrice', 'initialValue', 'currentValue', 'cashPnl', 'percentPnl', 'totalBought', 'realizedPnl', 'percentRealizedPnl', 'curPrice', 'redeemable', 'mergeable', 'title', 'slug', 'icon', 'eventId', 'eventSlug', 'outcome', 'outcomeIndex', 'oppositeOutcome', 'oppositeAsset', 'endDate', 'negativeRisk']
First record:
{
  "proxyWallet": "0x2005d16a84ceefa912d4e380cd32e7ff827875ea",
  "asset": "77985259937045308793816579916638590572189987943927681873921893711796592307973",
  "conditionId": "0xa9a9acb457b3567dd8d1cacaa400ae6aec8e63f858a4560afdb612161e43e57c",
  "size": 181369.1314,
  "avgPrice": 0.6281,
  "initialValue": 113919.221,
  "currentValue": 58944.9677,
  "cashPnl": -54974.2533,
  "percentPnl": -48.2572,
  "totalBought": 181684.2282,
  "realizedPnl": 0,
  "percentRealizedPnl": -48.3469,
  "curPrice": 0.325,
  "redeemable": false,
  "mergeable": true,
  "title": "Timberwolves vs. Nuggets",
  "slug"

### Values

In [5]:
values = {}

for name, address in WALLETS.items():
    data = get(DATA_API, "/value", params={"user": address})
    values[name] = data
    print(f"{name}: {json.dumps(data, indent=2)}")

rn1: [
  {
    "user": "0x2005d16a84ceefa912d4e380cd32e7ff827875ea",
    "value": 517349.941
  }
]
sovereign: [
  {
    "user": "0xee613b3fc183ee44f9da9c05f53e2da107e3debf",
    "value": 82537.4196
  }
]
swisstony: [
  {
    "user": "0x204f72f35326db932158cba6adff0b9a1da95e14",
    "value": 200664.8995
  }
]


### Activity

In [6]:
activity_sample = {}

for name, address in WALLETS.items():
    data = get(DATA_API, "/activity", params={
        "user":          address,
        "limit":         10,           # small on purpose — just inspecting shape
        "sortDirection": "DESC",       # most recent first
    })
    activity_sample[name] = data
    print(f"\n=== {name} — showing {len(data)} records ===")
    if data:
        print("Fields:", list(data[0].keys()))
        print("Most recent event:")
        print(json.dumps(data[0], indent=2))
        print(f"Oldest in this page:")
        print(json.dumps(data[-1], indent=2))


=== rn1 — showing 10 records ===
Fields: ['proxyWallet', 'timestamp', 'conditionId', 'type', 'size', 'usdcSize', 'transactionHash', 'price', 'asset', 'side', 'outcomeIndex', 'title', 'slug', 'icon', 'eventSlug', 'outcome', 'name', 'pseudonym', 'bio', 'profileImage', 'profileImageOptimized']
Most recent event:
{
  "proxyWallet": "0x2005d16a84ceefa912d4e380cd32e7ff827875ea",
  "timestamp": 1776748930,
  "conditionId": "0xf743001c011ce91856ef3eca6daacbce587a98d88ebbb8669c65437b4e1db06e",
  "type": "TRADE",
  "size": 4.02,
  "usdcSize": 1.8492,
  "transactionHash": "0x395afda3f5a3efe1b1842ab2a0ab3836ccd0331c980414e1748421de2c4598ac",
  "price": 0.46,
  "asset": "113848110479592031259253354608837956257747317426814833913131888568473157087516",
  "side": "BUY",
  "outcomeIndex": 1,
  "title": "Gwangju: Alex Bolt vs Ilia Simakin",
  "slug": "atp-bolt-simakin-2026-04-19",
  "icon": "https://polymarket-upload.s3.us-east-2.amazonaws.com/atp-tour-b4390c4fb8.jpg",
  "eventSlug": "atp-bolt-simakin-

### Erliest Activity

In [7]:
for name, address in WALLETS.items():
    data = get(DATA_API, "/activity", params={
        "user":          address,
        "limit":         1,
        "sortDirection": "ASC",   # flip to oldest first
    })
    if data:
        ts = data[0]["timestamp"]
        dt = datetime.fromtimestamp(ts, tz=timezone.utc)
        print(f"{name}: first activity {dt.strftime('%Y-%m-%d')}  (unix: {ts})")

rn1: first activity 2025-07-09  (unix: 1752062570)
sovereign: first activity 2025-08-07  (unix: 1754578485)
swisstony: first activity 2025-08-09  (unix: 1754716147)


## Full activity backfill with time-window chunking

In [8]:
from datetime import datetime, timezone, timedelta
MIN_WINDOW_SECONDS = 3600  # 1 hour floor — prevents infinite recursion

In [9]:
def fetch_window(address, win_start, win_end):
    """
    Fetch all records in a single time window via offset pagination.

    Returns (records, hit_cap).
    hit_cap=True means the window is too dense and should be split.
    We stop at offset 3000 (not 3500) to stay safely under the hard cap.
    """
    all_records = {}
    offset      = 0
    limit       = 500
    SAFE_CAP    = 3000

    while True:
        if offset > SAFE_CAP:
            return list(all_records.values()), True

        batch = get(DATA_API, "/activity", params={
            "user":          address,
            "start":         win_start,
            "end":           win_end,
            "limit":         limit,
            "offset":        offset,
            "sortDirection": "ASC",
        })

        if not batch:
            break

        for record in batch:
            key = record.get("transactionHash") or f"{record['timestamp']}_{record['conditionId']}"
            all_records[key] = record

        if len(batch) < limit:
            break
        
        time.sleep(0.5)
        offset += limit

    return list(all_records.values()), False

In [10]:
def fetch_recursive(address, win_start, win_end, depth=0):
    """
    Recursively fetch all activity in [win_start, win_end].

    If the window overflows the offset cap, bisect it and
    recurse into each half. Continues until windows fit
    or the minimum window size is reached.

    depth is tracked only for readable indentation in logs.
    """
    indent = "  " * (depth + 1)
    label  = datetime.fromtimestamp(win_start, tz=timezone.utc).strftime("%Y-%m-%d %H:%M")
    span   = win_end - win_start

    records, hit_cap = fetch_window(address, win_start, win_end)

    if not hit_cap:
        print(f"{indent}{label} (+{span//3600}h) | +{len(records)} records")
        return records

    # Window overflowed — check if we can still bisect
    if span <= MIN_WINDOW_SECONDS:
        print(f"{indent}⚠️  {label} | minimum window reached with overflow — data may be incomplete")
        return records

    # Bisect and recurse
    mid = (win_start + win_end) // 2
    print(f"{indent}{label} | overflow → bisecting into two {span//2//3600}h windows")

    left  = fetch_recursive(address, win_start, mid, depth + 1)
    right = fetch_recursive(address, mid, win_end, depth + 1)

    # Deduplicate across halves (bisect boundary could duplicate a record)
    merged = {}
    for record in left + right:
        key = record.get("transactionHash") or f"{record['timestamp']}_{record['conditionId']}"
        merged[key] = record

    return list(merged.values())

In [11]:
def fetch_activity_full(address, start_ts, end_ts, initial_window_days=7):
    """
    Fetch complete activity history for a wallet between start_ts and end_ts.

    Starts with weekly windows for efficiency, recursively bisects
    any window that overflows the API's offset cap.
    """
    all_records = {}

    # Carve the full range into initial weekly windows
    current = start_ts
    delta   = initial_window_days * 86400

    while current < end_ts:
        win_end = min(current + delta, end_ts)
        records = fetch_recursive(address, current, win_end, depth=0)

        for record in records:
            key = record.get("transactionHash") or f"{record['timestamp']}_{record['conditionId']}"
            all_records[key] = record

        current = win_end

    return list(all_records.values())

In [12]:
# Full history from each wallet's first known activity
WALLET_FIRST_TS = {
    "rn1":       1752062570,   # 2025-07-09
    "sovereign":  1754578485,   # 2025-08-07
    "swisstony":  1754716147,   # 2025-08-09
}

NOW_TS = int(datetime.now(tz=timezone.utc).timestamp())

In [14]:
all_activity = {}

for name, address in WALLETS.items():
    start_dt = datetime.fromtimestamp(WALLET_FIRST_TS[name], tz=timezone.utc).strftime("%Y-%m-%d")
    print(f"\n{'='*50}")
    print(f"Fetching {name} — full history from {start_dt}")
    records = fetch_activity_full(address, WALLET_FIRST_TS[name], NOW_TS)
    all_activity[name] = records
    print(f"→ {name} total: {len(records)} records\n")


Fetching rn1 — full history from 2025-07-09
  2025-07-09 12:02 (+168h) | +874 records
  2025-07-16 12:02 (+168h) | +1094 records
  2025-07-23 12:02 (+168h) | +1877 records
  2025-07-30 12:02 (+168h) | +1565 records
  2025-08-06 12:02 (+168h) | +2375 records
  2025-08-13 12:02 (+168h) | +2583 records
  2025-08-20 12:02 | overflow → bisecting into two 84h windows
    2025-08-20 12:02 (+84h) | +1855 records
    2025-08-24 00:02 (+84h) | +1754 records
  2025-08-27 12:02 | overflow → bisecting into two 84h windows
    2025-08-27 12:02 | overflow → bisecting into two 42h windows
      2025-08-27 12:02 (+42h) | +1109 records
      2025-08-29 06:02 | overflow → bisecting into two 21h windows
        2025-08-29 06:02 (+21h) | +1169 records
        2025-08-30 03:02 (+21h) | +2002 records
    2025-08-31 00:02 (+84h) | +2653 records
  2025-09-03 12:02 | overflow → bisecting into two 84h windows
    2025-09-03 12:02 (+84h) | +1969 records
    2025-09-07 00:02 (+84h) | +2803 records
  2025-09-10 12

KeyboardInterrupt: 